# SlideVQA Eval

Remote-only eval notebook for `/root/autodl-tmp/visual_rag_agent`.

Default eval is a 3-shard parallel run of the main typed-memory `Agent` method on official `NTT-hil-insight/SlideVQA` first-200 test data. Each shard loads the local Qwen3-VL retriever and VLM once, uses `flash_attention_2`, and optionally judges answers with DeepSeek V4 Flash via API. Deepspeed is not needed for eval.


## 1. Setup


In [1]:
from pathlib import Path
import importlib.metadata as metadata
import importlib.util
import json
import os
import subprocess
import sys
import time

PROJECT = Path('/root/autodl-tmp/visual_rag_agent')
os.chdir(PROJECT)
os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'
os.environ['PYTHONPATH'] = str(PROJECT)
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['OMP_NUM_THREADS'] = '8'
os.environ['DS_BUILD_OPS'] = '0'

print('cwd =', Path.cwd())
print('HF_ENDPOINT =', os.environ.get('HF_ENDPOINT'))
print('OMP_NUM_THREADS =', os.environ.get('OMP_NUM_THREADS'))

def load_env_file(path: Path):
    if not path.exists():
        return
    for raw_line in path.read_text(encoding='utf-8').splitlines():
        line = raw_line.strip()
        if not line or line.startswith('#') or '=' not in line:
            continue
        key, value = line.split('=', 1)
        os.environ.setdefault(key.strip(), value.strip().strip('\"').strip("'"))

load_env_file(PROJECT / '.env')
print('DEEPSEEK_API_KEY =', '<set>' if os.environ.get('DEEPSEEK_API_KEY') else '<missing>')
print('DEEPSEEK_MODEL =', os.environ.get('DEEPSEEK_MODEL', 'deepseek-v4-flash'))

def run(cmd, *, check=True):
    printable = ' '.join(str(x) for x in cmd) if isinstance(cmd, (list, tuple)) else str(cmd)
    print('\n$ ' + printable, flush=True)
    return subprocess.run(cmd, check=check)

def module_status(name):
    return 'FOUND' if importlib.util.find_spec(name) else 'MISSING'

def package_version(name):
    try:
        return metadata.version(name)
    except metadata.PackageNotFoundError:
        return 'not installed'

def latest_child(path: Path):
    children = [p for p in path.iterdir() if p.is_dir()] if path.exists() else []
    return max(children, key=lambda p: p.stat().st_mtime) if children else None


cwd = /root/autodl-tmp/visual_rag_agent
HF_ENDPOINT = https://hf-mirror.com
OMP_NUM_THREADS = 8
DEEPSEEK_API_KEY = <set>
DEEPSEEK_MODEL = deepseek-v4-flash


## 2. Verify Runtime


In [2]:
import torch

print('python:', sys.version.split()[0])
print('torch:', torch.__version__, 'cuda:', torch.version.cuda)
print('cuda_available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))

for name in ['flash_attn', 'flash_attn_2_cuda', 'transformers', 'qwen_vl_utils', 'accelerate', 'yaml']:
    print(f'{name}:', module_status(name))

import flash_attn, flash_attn_2_cuda, transformers
print('flash_attn_version:', getattr(flash_attn, '__version__', 'unknown'))
print('transformers_version:', transformers.__version__)
print('note: deepspeed is intentionally not used in this eval notebook; handle it later for training.')


python: 3.12.3
torch: 2.8.0+cu128 cuda: 12.8
cuda_available: True
gpu: NVIDIA RTX PRO 6000 Blackwell Server Edition
flash_attn: FOUND
flash_attn_2_cuda: FOUND
transformers: FOUND
qwen_vl_utils: FOUND
accelerate: FOUND
yaml: FOUND
flash_attn_version: 2.8.3
transformers_version: 5.9.0
note: deepspeed is intentionally not used in this eval notebook; handle it later for training.


## 3. Paths And Data Checks


In [3]:
DATASET_FILE = PROJECT / 'data/corpora/slidevqa/test.jsonl'
CORPUS_DIR = PROJECT / 'data/corpora/slidevqa/pages'
INDEX_DIR = PROJECT / 'data/indexes/slidevqa'
OUTPUT_ROOT = PROJECT / 'outputs/eval_notebook'
CONFIG_PATH = PROJECT / 'config/eval_flash.yaml'

LOCAL_VLM = Path('/root/autodl-tmp/models/Qwen3-VL-4B-Instruct')
LOCAL_RETRIEVER = Path('/root/autodl-tmp/models/Qwen3-VL-Embedding-8B')
VLM_MODEL = str(LOCAL_VLM)
RETRIEVER_MODEL = str(LOCAL_RETRIEVER)

checks = {
    'dataset_file': DATASET_FILE.exists(),
    'corpus_dir': CORPUS_DIR.exists(),
    'index_embeddings': (INDEX_DIR / 'embeddings.npy').exists(),
    'index_filenames': (INDEX_DIR / 'filenames.json').exists(),
    'vlm_model': LOCAL_VLM.exists(),
    'retriever_model': LOCAL_RETRIEVER.exists(),
}
print(json.dumps({k: str(v) if isinstance(v, Path) else v for k, v in {
    'DATASET_FILE': DATASET_FILE,
    'CORPUS_DIR': CORPUS_DIR,
    'INDEX_DIR': INDEX_DIR,
    'OUTPUT_ROOT': OUTPUT_ROOT,
    'CONFIG_PATH': CONFIG_PATH,
    'VLM_MODEL': VLM_MODEL,
    'RETRIEVER_MODEL': RETRIEVER_MODEL,
    'checks': checks,
}.items()}, ensure_ascii=False, indent=2))

missing = [name for name, ok in checks.items() if not ok]
if missing:
    raise FileNotFoundError('Missing required remote eval assets: ' + ', '.join(missing))

rows = []
with DATASET_FILE.open('r', encoding='utf-8') as f:
    for line in f:
        if line.strip():
            rows.append(json.loads(line))
print('dataset_rows:', len(rows))
print('page_images:', sum(1 for _ in CORPUS_DIR.rglob('*.png')))
print('first_sample:', json.dumps({
    'id': rows[0].get('id'),
    'question': rows[0].get('question'),
    'answer': rows[0].get('answer'),
    'evidence_pages': rows[0].get('evidence_pages'),
    'page_images_count': len(rows[0].get('page_images', [])),
}, ensure_ascii=False, indent=2))


{
  "DATASET_FILE": "/root/autodl-tmp/visual_rag_agent/data/corpora/slidevqa/test.jsonl",
  "CORPUS_DIR": "/root/autodl-tmp/visual_rag_agent/data/corpora/slidevqa/pages",
  "INDEX_DIR": "/root/autodl-tmp/visual_rag_agent/data/indexes/slidevqa",
  "OUTPUT_ROOT": "/root/autodl-tmp/visual_rag_agent/outputs/eval_notebook",
  "CONFIG_PATH": "/root/autodl-tmp/visual_rag_agent/config/eval_flash.yaml",
  "VLM_MODEL": "/root/autodl-tmp/models/Qwen3-VL-4B-Instruct",
  "RETRIEVER_MODEL": "/root/autodl-tmp/models/Qwen3-VL-Embedding-8B",
  "checks": {
    "dataset_file": true,
    "corpus_dir": true,
    "index_embeddings": true,
    "index_filenames": true,
    "vlm_model": true,
    "retriever_model": true
  }
}
dataset_rows: 200
page_images: 900
first_sample: {
  "id": 0,
  "question": "How much is the Trading Operating Profit in 2011?",
  "answer": "12.5 bn",
  "evidence_pages": [
    5
  ],
  "page_images_count": 20
}


## 4. Write Eval Config


In [4]:
import yaml

CONFIG_PATH.parent.mkdir(parents=True, exist_ok=True)
config = {
    'models': {
        'vlm': {
            'provider': 'qwen',
            'name': VLM_MODEL,
            'max_tokens': 1024,
            'temperature': 0.0,
        },
        'retriever': {
            'name': RETRIEVER_MODEL,
            'index_path': str(INDEX_DIR),
        },
    },
    'agent': {
        'top_k': 1,
        'max_iters': 5,
        'partial_memory_capacity': 2,
    },
    'image_budget': {
        'yes_pixels': 400000,
        'partial_pixels': 200000,
    },
    'dataset': {
        'name': 'slidevqa',
        'split': 'test',
        'num_samples': 200,
    },
    'runtime': {
        'output_dir': str(OUTPUT_ROOT),
        'device': 'cuda',
        'dtype': 'bfloat16',
        'attn_implementation': 'flash_attention_2',
    },
}
CONFIG_PATH.write_text(yaml.safe_dump(config, sort_keys=False), encoding='utf-8')
print(CONFIG_PATH.read_text(encoding='utf-8'))


models:
  vlm:
    provider: qwen
    name: /root/autodl-tmp/models/Qwen3-VL-4B-Instruct
    max_tokens: 1024
    temperature: 0.0
  retriever:
    name: /root/autodl-tmp/models/Qwen3-VL-Embedding-8B
    index_path: /root/autodl-tmp/visual_rag_agent/data/indexes/slidevqa
agent:
  top_k: 1
  max_iters: 5
  partial_memory_capacity: 2
image_budget:
  yes_pixels: 400000
  partial_pixels: 200000
dataset:
  name: slidevqa
  split: test
  num_samples: 200
runtime:
  output_dir: /root/autodl-tmp/visual_rag_agent/outputs/eval_notebook
  device: cuda
  dtype: bfloat16
  attn_implementation: flash_attention_2



## 5. Optional One-Sample Smoke Test

Disabled by default to avoid an extra model load before the real eval.


In [5]:
RUN_ONE_SAMPLE_SMOKE = False
JUDGE_WITH_DEEPSEEK = True
JUDGE_MODEL = os.environ.get('DEEPSEEK_MODEL', 'deepseek-v4-flash')

if RUN_ONE_SAMPLE_SMOKE:
    smoke_output = OUTPUT_ROOT / 'main_smoke'
    cmd = [
        sys.executable, 'scripts/evaluate.py',
        '--dataset-file', str(DATASET_FILE),
        '--num-samples', '1',
        '--config', str(CONFIG_PATH),
        '--index', str(INDEX_DIR),
        '--output', str(smoke_output),
    ]
    if JUDGE_WITH_DEEPSEEK:
        cmd += ['--judge', 'deepseek', '--judge-model', JUDGE_MODEL]
    run(cmd)
    smoke_run = latest_child(smoke_output)
    print('smoke_run:', smoke_run)
    pred_path = smoke_run / 'predictions.jsonl'
    pred = json.loads(pred_path.read_text(encoding='utf-8').splitlines()[0])
    print(json.dumps(pred, ensure_ascii=False, indent=2))


## 6. Launch First-200 Eval In 3 Parallel Shards

This is the default eval path. It starts three background processes: samples 0-66, 67-133, and 134-199. The same cell can follow all logs and prints only one line per finished sample, e.g. `[part0] [eval] step=1/67 sample_id=0 terminated_by=decide`.


In [6]:
import shlex

RUN_PARALLEL_AGENT_FIRST200 = True
JUDGE_WITH_DEEPSEEK = True
JUDGE_MODEL = os.environ.get('DEEPSEEK_MODEL', 'deepseek-v4-flash')
KILL_EXISTING_EVAL = True
CLEAN_SHARD_LOGS = True
FOLLOW_LOGS_AFTER_LAUNCH = True
POLL_SECONDS = 5
LAUNCH_STAGGER_SECONDS = 0

PARALLEL_ROOT = PROJECT / 'outputs/eval_parallel3'
PARALLEL_LOG_ROOT = PROJECT / 'logs/eval_parallel3'
SHARDS = [
    {'name': 'part0', 'start': 0, 'n': 67},
    {'name': 'part1', 'start': 67, 'n': 67},
    {'name': 'part2', 'start': 134, 'n': 66},
]

def current_eval_pids() -> list[str]:
    result = subprocess.run(['ps', '-eo', 'pid=,args='], capture_output=True, text=True, check=True)
    pids = []
    for line in result.stdout.splitlines():
        parts = line.strip().split(None, 1)
        if len(parts) != 2:
            continue
        pid, args = parts
        if 'scripts/evaluate.py' in args and 'python' in args:
            pids.append(pid)
    return pids

def eval_pid_running(pid: str) -> bool:
    if not pid:
        return False
    result = subprocess.run(['ps', '-p', pid, '-o', 'args='], capture_output=True, text=True, check=False)
    return result.returncode == 0 and 'scripts/evaluate.py' in result.stdout

def kill_existing_eval() -> None:
    pids = current_eval_pids()
    if not pids:
        print('no existing evaluate.py process')
        return
    print('killing existing evaluate.py pids:', ' '.join(pids))
    subprocess.run(['kill', *pids], check=False)
    deadline = time.time() + 10
    while time.time() < deadline:
        remaining = current_eval_pids()
        if not remaining:
            return
        time.sleep(1)
    remaining = current_eval_pids()
    if remaining:
        print('force killing remaining evaluate.py pids:', ' '.join(remaining))
        subprocess.run(['kill', '-9', *remaining], check=False)
        time.sleep(1)

def clean_shard_state() -> None:
    PARALLEL_LOG_ROOT.mkdir(parents=True, exist_ok=True)
    for shard in SHARDS:
        name = shard['name']
        log_path = PARALLEL_LOG_ROOT / f'{name}.log'
        pid_path = PARALLEL_LOG_ROOT / f'{name}.pid'
        if pid_path.exists() and eval_pid_running(pid_path.read_text().strip()):
            raise RuntimeError(f'{name} is still running; kill it before relaunching')
        if CLEAN_SHARD_LOGS:
            log_path.unlink(missing_ok=True)
            pid_path.unlink(missing_ok=True)

def eval_log_lines(path: Path) -> list[str]:
    if not path.exists():
        return []
    return [line for line in path.read_text(encoding='utf-8', errors='replace').splitlines() if line.startswith('[eval]')]

def launch_shard(shard: dict) -> None:
    name = shard['name']
    log_path = PARALLEL_LOG_ROOT / f'{name}.log'
    pid_path = PARALLEL_LOG_ROOT / f'{name}.pid'
    out_path = PARALLEL_ROOT / f'main_agent_{name}'
    cmd = [
        sys.executable, 'scripts/evaluate.py',
        '--dataset-file', str(DATASET_FILE),
        '--start-index', str(shard['start']),
        '--num-samples', str(shard['n']),
        '--config', str(CONFIG_PATH),
        '--index', str(INDEX_DIR),
        '--output', str(out_path),
    ]
    if JUDGE_WITH_DEEPSEEK:
        cmd += ['--judge', 'deepseek', '--judge-model', JUDGE_MODEL]
    env_cmd = [
        'env',
        'OMP_NUM_THREADS=6',
        'PYTHONPATH=.',
        'PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True',
        'HF_HUB_DISABLE_PROGRESS_BARS=1',
        'TRANSFORMERS_VERBOSITY=error',
        *cmd,
    ]
    shell_cmd = (
        'cd ' + shlex.quote(str(PROJECT)) + ' && '
        + 'nohup ' + ' '.join(shlex.quote(str(x)) for x in env_cmd)
        + ' > ' + shlex.quote(str(log_path))
        + ' 2>&1 & echo $! > ' + shlex.quote(str(pid_path))
    )
    subprocess.run(['bash', '-lc', shell_cmd], check=True)
    print(f"{name} pid={pid_path.read_text().strip()} log={log_path}")

if RUN_PARALLEL_AGENT_FIRST200:
    PARALLEL_LOG_ROOT.mkdir(parents=True, exist_ok=True)
    if KILL_EXISTING_EVAL:
        kill_existing_eval()
    clean_shard_state()

    for shard_idx, shard in enumerate(SHARDS):
        launch_shard(shard)
        if shard_idx < len(SHARDS) - 1 and LAUNCH_STAGGER_SECONDS > 0:
            time.sleep(LAUNCH_STAGGER_SECONDS)

    if FOLLOW_LOGS_AFTER_LAUNCH:
        seen = {shard['name']: 0 for shard in SHARDS}
        while True:
            any_running = False
            for shard in SHARDS:
                name = shard['name']
                log_path = PARALLEL_LOG_ROOT / f'{name}.log'
                pid_path = PARALLEL_LOG_ROOT / f'{name}.pid'
                lines = eval_log_lines(log_path)
                for line in lines[seen[name]:]:
                    print(f'[{name}] {line}', flush=True)
                seen[name] = len(lines)
                if pid_path.exists() and eval_pid_running(pid_path.read_text().strip()):
                    any_running = True
            if not any_running:
                print('parallel eval finished')
                break
            time.sleep(POLL_SECONDS)


no existing evaluate.py process
part0 pid=1823 log=/root/autodl-tmp/visual_rag_agent/logs/eval_parallel3/part0.log
part1 pid=1828 log=/root/autodl-tmp/visual_rag_agent/logs/eval_parallel3/part1.log
part2 pid=1833 log=/root/autodl-tmp/visual_rag_agent/logs/eval_parallel3/part2.log
[part0] [eval] step=1/67 sample_id=0 terminated_by=decide
[part2] [eval] step=1/66 sample_id=134 terminated_by=decide
[part2] [eval] step=2/66 sample_id=135 terminated_by=decide
[part1] [eval] step=1/67 sample_id=67 terminated_by=max_iters
[part0] [eval] step=2/67 sample_id=1 terminated_by=decide
[part2] [eval] step=3/66 sample_id=136 terminated_by=decide
[part1] [eval] step=2/67 sample_id=68 terminated_by=decide
[part0] [eval] step=3/67 sample_id=2 terminated_by=decide
[part2] [eval] step=4/66 sample_id=137 terminated_by=decide
[part2] [eval] step=5/66 sample_id=138 terminated_by=decide
[part2] [eval] step=6/66 sample_id=139 terminated_by=decide
[part1] [eval] step=3/67 sample_id=69 terminated_by=max_iters
[p

## 7. Check Parallel Eval Progress


In [7]:
PARALLEL_ROOT = PROJECT / 'outputs/eval_parallel3'
PARALLEL_LOG_ROOT = PROJECT / 'logs/eval_parallel3'
SHARDS = [
    {'name': 'part0', 'expected': 67},
    {'name': 'part1', 'expected': 67},
    {'name': 'part2', 'expected': 66},
]

for shard in SHARDS:
    name = shard['name']
    pid_path = PARALLEL_LOG_ROOT / f'{name}.pid'
    log_path = PARALLEL_LOG_ROOT / f'{name}.log'
    out_root = PARALLEL_ROOT / f'main_agent_{name}'
    print()
    print('===', name, '===')
    if pid_path.exists():
        pid = pid_path.read_text().strip()
        print('pid:', pid)
        run(['bash', '-lc', f'ps -fp {pid} || true'])
    if log_path.exists():
        print('log:', log_path)
        run(['bash', '-lc', f'tail -n 20 {log_path}'])
    run([sys.executable, 'scripts/eval_progress.py', '--root', str(out_root), '--expected', str(shard['expected'])], check=False)



=== part0 ===
pid: 1823

$ bash -lc ps -fp 1823 || true
UID         PID   PPID  C STIME TTY          TIME CMD
log: /root/autodl-tmp/visual_rag_agent/logs/eval_parallel3/part0.log

$ bash -lc tail -n 20 /root/autodl-tmp/visual_rag_agent/logs/eval_parallel3/part0.log
    "warning_utilization": {
      "search_queries": 106,
      "warning_informed_queries": 34,
      "fraction": 0.32075471698113206
    },
    "avg_visual_pixels_at_answer": 346268.6567164179,
    "failure_type_counts": {
      "null": 42
    },
    "accuracy": 0.7761194029850746,
    "judge": {
      "provider": "deepseek",
      "model": "deepseek-v4-flash",
      "judged": 67,
      "correct": 52,
      "errors": 0,
      "accuracy": 0.7761194029850746
    }
  }
}

$ /root/miniconda3/bin/python scripts/eval_progress.py --root /root/autodl-tmp/visual_rag_agent/outputs/eval_parallel3/main_agent_part0 --expected 67
{
  "run": "/root/autodl-tmp/visual_rag_agent/outputs/eval_parallel3/main_agent_part0/20260524_182515",
  "e

## 8. Merge Completed Shards


In [9]:
MERGE_PARALLEL_SHARDS = True

if MERGE_PARALLEL_SHARDS:
    run([
        sys.executable, 'scripts/merge_eval_shards.py',
        '--roots',
        str(PARALLEL_ROOT / 'main_agent_part0'),
        str(PARALLEL_ROOT / 'main_agent_part1'),
        str(PARALLEL_ROOT / 'main_agent_part2'),
        '--output', str(PARALLEL_ROOT / 'merged'),
        '--baseline', 'agent',
    ])



$ /root/miniconda3/bin/python scripts/merge_eval_shards.py --roots /root/autodl-tmp/visual_rag_agent/outputs/eval_parallel3/main_agent_part0 /root/autodl-tmp/visual_rag_agent/outputs/eval_parallel3/main_agent_part1 /root/autodl-tmp/visual_rag_agent/outputs/eval_parallel3/main_agent_part2 --output /root/autodl-tmp/visual_rag_agent/outputs/eval_parallel3/merged --baseline agent
{
  "output": "/root/autodl-tmp/visual_rag_agent/outputs/eval_parallel3/merged/20260524_185959",
  "num_rows": 200,
  "summary": {
    "num_samples": 200,
    "baseline": "agent",
    "mean_iters": 1.645,
    "median_iters": 1.0,
    "termination_counts": {
      "decide": 172,
      "max_iters": 28
    },
    "decision_distribution": {
      "yes": 177,
      "partial": 51,
      "no": 101
    },
    "warning_utilization": {
      "search_queries": 329,
      "warning_informed_queries": 85,
      "fraction": 0.25835866261398177
    },
    "avg_visual_pixels_at_answer": 390000,
    "failure_type_counts": {
      

## 9. Launch Baseline First-200 Eval In 3 Parallel Shards

Agentic-summary baseline: decide sees all accumulated page summaries plus recent two retrieval rounds of images; search result pages are summarized into query memory. This is separate from the main typed-memory Agent cells above.


In [11]:
import shlex

RUN_PARALLEL_AGENTIC_SUMMARY_FIRST200 = True
BASELINE_JUDGE_WITH_DEEPSEEK = True
BASELINE_JUDGE_MODEL = os.environ.get('DEEPSEEK_MODEL', 'deepseek-v4-flash')
BASELINE_KILL_EXISTING_EVAL = True
BASELINE_CLEAN_SHARD_LOGS = True
BASELINE_FOLLOW_LOGS_AFTER_LAUNCH = True
BASELINE_POLL_SECONDS = 5
BASELINE_LAUNCH_STAGGER_SECONDS = 0

BASELINE_PARALLEL_ROOT = PROJECT / 'outputs/eval_baseline_parallel3'
BASELINE_PARALLEL_LOG_ROOT = PROJECT / 'logs/eval_baseline_parallel3'
BASELINE_SHARDS = [
    {'name': 'part0', 'start': 0, 'n': 67},
    {'name': 'part1', 'start': 67, 'n': 67},
    {'name': 'part2', 'start': 134, 'n': 66},
]

def baseline_clean_shard_state() -> None:
    BASELINE_PARALLEL_LOG_ROOT.mkdir(parents=True, exist_ok=True)
    for shard in BASELINE_SHARDS:
        name = shard['name']
        log_path = BASELINE_PARALLEL_LOG_ROOT / f'{name}.log'
        pid_path = BASELINE_PARALLEL_LOG_ROOT / f'{name}.pid'
        if pid_path.exists() and eval_pid_running(pid_path.read_text().strip()):
            raise RuntimeError(f'baseline {name} is still running; kill it before relaunching')
        if BASELINE_CLEAN_SHARD_LOGS:
            log_path.unlink(missing_ok=True)
            pid_path.unlink(missing_ok=True)

def launch_baseline_shard(shard: dict) -> None:
    name = shard['name']
    log_path = BASELINE_PARALLEL_LOG_ROOT / f'{name}.log'
    pid_path = BASELINE_PARALLEL_LOG_ROOT / f'{name}.pid'
    out_path = BASELINE_PARALLEL_ROOT / f'agentic_summary_{name}'
    cmd = [
        sys.executable, 'scripts/evaluate.py',
        '--dataset-file', str(DATASET_FILE),
        '--start-index', str(shard['start']),
        '--num-samples', str(shard['n']),
        '--config', str(CONFIG_PATH),
        '--index', str(INDEX_DIR),
        '--output', str(out_path),
        '--baseline', 'agentic_summary',
    ]
    if BASELINE_JUDGE_WITH_DEEPSEEK:
        cmd += ['--judge', 'deepseek', '--judge-model', BASELINE_JUDGE_MODEL]
    env_cmd = [
        'env',
        'OMP_NUM_THREADS=6',
        'PYTHONPATH=.',
        'PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True',
        'HF_HUB_DISABLE_PROGRESS_BARS=1',
        'TRANSFORMERS_VERBOSITY=error',
        *cmd,
    ]
    shell_cmd = (
        'cd ' + shlex.quote(str(PROJECT)) + ' && '
        + 'nohup ' + ' '.join(shlex.quote(str(x)) for x in env_cmd)
        + ' > ' + shlex.quote(str(log_path))
        + ' 2>&1 & echo $! > ' + shlex.quote(str(pid_path))
    )
    subprocess.run(['bash', '-lc', shell_cmd], check=True)
    print(f"baseline {name} pid={pid_path.read_text().strip()} log={log_path}")

if RUN_PARALLEL_AGENTIC_SUMMARY_FIRST200:
    BASELINE_PARALLEL_LOG_ROOT.mkdir(parents=True, exist_ok=True)
    if BASELINE_KILL_EXISTING_EVAL:
        kill_existing_eval()
    baseline_clean_shard_state()

    for shard_idx, shard in enumerate(BASELINE_SHARDS):
        launch_baseline_shard(shard)
        if shard_idx < len(BASELINE_SHARDS) - 1 and BASELINE_LAUNCH_STAGGER_SECONDS > 0:
            time.sleep(BASELINE_LAUNCH_STAGGER_SECONDS)

    if BASELINE_FOLLOW_LOGS_AFTER_LAUNCH:
        seen = {shard['name']: 0 for shard in BASELINE_SHARDS}
        while True:
            any_running = False
            for shard in BASELINE_SHARDS:
                name = shard['name']
                log_path = BASELINE_PARALLEL_LOG_ROOT / f'{name}.log'
                pid_path = BASELINE_PARALLEL_LOG_ROOT / f'{name}.pid'
                lines = eval_log_lines(log_path)
                for line in lines[seen[name]:]:
                    print(f'[baseline {name}] {line}', flush=True)
                seen[name] = len(lines)
                if pid_path.exists() and eval_pid_running(pid_path.read_text().strip()):
                    any_running = True
            if not any_running:
                print('baseline parallel eval finished')
                break
            time.sleep(BASELINE_POLL_SECONDS)


no existing evaluate.py process
baseline part0 pid=4101 log=/root/autodl-tmp/visual_rag_agent/logs/eval_baseline_parallel3/part0.log
baseline part1 pid=4106 log=/root/autodl-tmp/visual_rag_agent/logs/eval_baseline_parallel3/part1.log
baseline part2 pid=4111 log=/root/autodl-tmp/visual_rag_agent/logs/eval_baseline_parallel3/part2.log
[baseline part2] [eval] step=1/66 sample_id=134 terminated_by=decide
[baseline part0] [eval] step=1/67 sample_id=0 terminated_by=decide
[baseline part2] [eval] step=2/66 sample_id=135 terminated_by=max_iters
[baseline part1] [eval] step=1/67 sample_id=67 terminated_by=max_iters
[baseline part0] [eval] step=2/67 sample_id=1 terminated_by=max_iters
[baseline part1] [eval] step=2/67 sample_id=68 terminated_by=decide
[baseline part2] [eval] step=3/66 sample_id=136 terminated_by=decide
[baseline part0] [eval] step=3/67 sample_id=2 terminated_by=decide
[baseline part2] [eval] step=4/66 sample_id=137 terminated_by=decide
[baseline part2] [eval] step=5/66 sample_id

## 10. Check Baseline Parallel Eval Progress


In [12]:
BASELINE_PARALLEL_ROOT = PROJECT / 'outputs/eval_baseline_parallel3'
BASELINE_PARALLEL_LOG_ROOT = PROJECT / 'logs/eval_baseline_parallel3'
BASELINE_SHARDS = [
    {'name': 'part0', 'expected': 67},
    {'name': 'part1', 'expected': 67},
    {'name': 'part2', 'expected': 66},
]

for shard in BASELINE_SHARDS:
    name = shard['name']
    pid_path = BASELINE_PARALLEL_LOG_ROOT / f'{name}.pid'
    log_path = BASELINE_PARALLEL_LOG_ROOT / f'{name}.log'
    out_root = BASELINE_PARALLEL_ROOT / f'agentic_summary_{name}'
    print()
    print('=== baseline', name, '===')
    if pid_path.exists():
        pid = pid_path.read_text().strip()
        print('pid:', pid)
        run(['bash', '-lc', f'ps -fp {pid} || true'])
    if log_path.exists():
        print('log:', log_path)
        run(['bash', '-lc', f'tail -n 20 {log_path}'])
    run([sys.executable, 'scripts/eval_progress.py', '--root', str(out_root), '--expected', str(shard['expected'])], check=False)



=== baseline part0 ===
pid: 4101

$ bash -lc ps -fp 4101 || true
UID         PID   PPID  C STIME TTY          TIME CMD
log: /root/autodl-tmp/visual_rag_agent/logs/eval_baseline_parallel3/part0.log

$ bash -lc tail -n 20 /root/autodl-tmp/visual_rag_agent/logs/eval_baseline_parallel3/part0.log
    "mean_iters": 1.4776119402985075,
    "median_iters": 1,
    "termination_counts": {
      "decide": 59,
      "max_iters": 8
    },
    "mean_page_summaries": 1.4776119402985075,
    "search_queries": 99,
    "summarise_calls": 99,
    "accuracy": 0.7164179104477612,
    "judge": {
      "provider": "deepseek",
      "model": "deepseek-v4-flash",
      "judged": 67,
      "correct": 48,
      "errors": 0,
      "accuracy": 0.7164179104477612
    }
  }
}

$ /root/miniconda3/bin/python scripts/eval_progress.py --root /root/autodl-tmp/visual_rag_agent/outputs/eval_baseline_parallel3/agentic_summary_part0 --expected 67
{
  "run": "/root/autodl-tmp/visual_rag_agent/outputs/eval_baseline_parallel3/

## 11. Merge Completed Baseline Shards


In [15]:
MERGE_BASELINE_PARALLEL_SHARDS = True

if MERGE_BASELINE_PARALLEL_SHARDS:
    run([
        sys.executable, 'scripts/merge_eval_shards.py',
        '--roots',
        str(BASELINE_PARALLEL_ROOT / 'agentic_summary_part0'),
        str(BASELINE_PARALLEL_ROOT / 'agentic_summary_part1'),
        str(BASELINE_PARALLEL_ROOT / 'agentic_summary_part2'),
        '--output', str(BASELINE_PARALLEL_ROOT / 'merged'),
        '--baseline', 'agentic_summary',
    ])



$ /root/miniconda3/bin/python scripts/merge_eval_shards.py --roots /root/autodl-tmp/visual_rag_agent/outputs/eval_baseline_parallel3/agentic_summary_part0 /root/autodl-tmp/visual_rag_agent/outputs/eval_baseline_parallel3/agentic_summary_part1 /root/autodl-tmp/visual_rag_agent/outputs/eval_baseline_parallel3/agentic_summary_part2 --output /root/autodl-tmp/visual_rag_agent/outputs/eval_baseline_parallel3/merged --baseline agentic_summary
{
  "output": "/root/autodl-tmp/visual_rag_agent/outputs/eval_baseline_parallel3/merged/20260524_193323",
  "num_rows": 200,
  "summary": {
    "num_samples": 200,
    "baseline": "agentic_summary",
    "mean_iters": 1.515,
    "median_iters": 1.0,
    "termination_counts": {
      "decide": 176,
      "max_iters": 24
    },
    "mean_page_summaries": 1.515,
    "search_queries": 303,
    "summarise_calls": 303,
    "accuracy": 0.68,
    "judge": {
      "provider": "deepseek",
      "model": "deepseek-v4-flash",
      "judged": 200,
      "correct": 13